# 02_causal_forest — τ(x): retorno marginal heterogéneo del gasto

Cruza `residuo_Y` (de `01_anemia_model.ipynb`) y `residuo_T` (de `01_spending_model.ipynb`) por `ubigeo + anio`, y entrena un **Causal Forest** (honest splitting) sobre esos residuos ya limpios de confusión territorial, usando `CausalForestDML` de EconML.

**Importante — léase antes de presentar los resultados:** el hallazgo principal de esta corrida no es el que uno esperaría, y hay que comunicarlo con la misma honestidad con la que se documentó todo lo demás. Ver la sección 5 antes de armar cualquier slide con estos números.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.dummy import DummyRegressor
from econml.dml import CausalForestDML

anemia = pd.read_csv("data/predictions/residuos_anemia_model.csv")
spend = pd.read_csv("data/predictions/residuos_spending_model.csv")
df = pd.read_csv("data/clean/merged/causal_model_data.csv")
df = df[df["anio"].between(2021, 2025)].copy()

print(f"residuo_Y: {len(anemia)} filas")
print(f"residuo_T: {len(spend)} filas")


## 1. Cruce por `ubigeo + anio`

Los dos modelos excluyeron filas distintas (anemia excluyó 146, gasto excluyó 239, sin superposición exacta), así que el `inner join` se queda solo con las filas que tienen **ambos** residuos disponibles — va a ser un poco menos que las 9,206 de `spending_model` porque ese ya era el subconjunto más chico.

In [ ]:
cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

merged = (
    anemia[["ubigeo", "anio", "residuo_Y"]]
    .merge(spend[["ubigeo", "anio", "residuo_T"]], on=["ubigeo", "anio"], how="inner")
    .merge(df[["ubigeo", "anio", "distrito", "departamento"] + cols_X], on=["ubigeo", "anio"], how="inner")
)

assert merged[cols_X + ["residuo_Y", "residuo_T"]].isnull().sum().sum() == 0, "hay nulos en el cruce"

print(f"Filas cruzadas (con residuo_Y y residuo_T disponibles): {len(merged)}")
print(f"Distritos únicos: {merged['ubigeo'].nunique()}")


## 2. Entrenamiento del Causal Forest

Se usa `CausalForestDML` de EconML, pero **sin dejar que vuelva a residualizar** — `residuo_Y` y `residuo_T` ya sallieron de un cross-fitting propio (LightGBM + Optuna, `GroupKFold` por distrito) en los dos notebooks anteriores. Para que `CausalForestDML` no le reste una segunda vez su propio modelo interno, se le pasa `model_y` y `model_t` como un regresor que siempre predice 0 (`DummyRegressor`) — así, el residuo que usa internamente para entrenar el bosque es exactamente `residuo_Y - 0 = residuo_Y` (y lo mismo para T), sin duplicar el trabajo ni introducir un segundo modelo de nuisance.

`honest=True` (por defecto) implementa el honest splitting: en cada árbol, una mitad de la sub-muestra decide los cortes de X, la otra mitad calcula el τ real de cada hoja.

In [ ]:
X = merged[cols_X].values
Y_res = merged["residuo_Y"].values
T_res = merged["residuo_T"].values

cf = CausalForestDML(
    model_y=DummyRegressor(strategy="constant", constant=0),
    model_t=DummyRegressor(strategy="constant", constant=0),
    n_estimators=1000,
    min_samples_leaf=10,
    honest=True,
    cv=2,
    random_state=42,
    n_jobs=-1,
)
cf.fit(Y_res, T_res, X=X)

tau = cf.effect(X)
lb, ub = cf.effect_interval(X, alpha=0.05)

print("τ(x) por fila (distrito-año) — estadísticas:")
print(pd.Series(tau).describe())


## 3. ATE global y chequeo del signo

Esto es lo que hay que leer con cuidado antes de la presentación.

In [ ]:
ate = cf.ate(X)
print(f"ATE global (efecto promedio): {ate:.4f}")
print()
print(cf.ate_inference(X).summary())
print()
print(f"Filas con τ negativo: {(tau < 0).sum()} de {len(tau)}")
print(f"Filas con intervalo de confianza que NO cruza cero (significativas): {(((lb>0)|(ub<0))).sum()} de {len(tau)}")


## 5. Cómo leer este resultado — antes de poner esto en un slide

**El ATE salió positivo (~0.017-0.018) y estadísticamente significativo, y NINGUNA fila tiene τ negativo.** Esto significa que, incluso después de restarle a Y y a T lo que el territorio (X) explica, la parte que queda del gasto sigue moviéndose en la misma dirección que la parte que queda de la anemia — más gasto residual va con más anemia residual, no menos.

**Esto casi seguro NO significa que gastar más cause más anemia.** La explicación más probable es la misma que motivó todo este proyecto: el Estado dirige el gasto del PAN hacia distritos con más anemia real — y esa focalización se basa en información (reportes de campo, decisiones de gestión) que el contexto territorial visible desde satélite no captura del todo. El modelo `T~X` no pudo "ver" esa señal de focalización porque no está en X, así que sobrevive en `residuo_T`, correlacionada con `residuo_Y` por la misma razón de fondo.

Esto no es un error del código — es exactamente la limitación que tu propio documento metodológico ya anticipó: *"Sin una fuente de variación exógena confirmada (se explora el canon minero como instrumento), τ se comunica como índice de priorización basado en heterogeneidad estimada, no como efecto causal probado"*. Sin ese instrumento (todavía no implementado), no hay forma honesta de aislar el efecto causal puro de la focalización administrativa.

**Qué SÍ puedes presentar con esto, de forma defendible:**
- La heterogeneidad entre distritos es real y varía bastante (τ va de ~0.0001 a ~0.047) — algunos distritos muestran una asociación gasto-anemia mucho más fuerte que otros, y eso sigue siendo información útil para priorizar.
- Encuadra τ explícitamente como **índice de priorización basado en heterogeneidad**, nunca como "el gasto reduce la anemia en X puntos" — tu propia metodología ya te da el lenguaje correcto para esto.
- Declara abiertamente esta limitación en la pantalla de metodología de la app — es exactamente el tipo de honestidad que tu documento de responsabilidad (sección 8) pide.

**Qué evitar:** no digas en la presentación "encontramos que el gasto reduce la anemia en tanto" — los datos, tal como están hoy (sin instrumento causal), no lo sostienen. Lo correcto es "encontramos que el retorno aparente del gasto varía mucho entre distritos, y ese patrón de heterogeneidad es lo que usamos para priorizar — la identificación causal pura queda como trabajo futuro con un instrumento validado".

## 6. Guardar resultados en `data/predictions/`

Dos archivos: uno a nivel distrito-año (el que entrenó el bosque) y uno agregado a nivel distrito (promediando X de los años disponibles y prediciendo τ en ese punto) — este segundo es el que alimenta el mapa final de la app, que necesita un solo número por distrito.

In [ ]:
OUTPUT_DIR = Path("data/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# nivel distrito-año
salida_panel = merged[["ubigeo", "anio", "distrito", "departamento", "residuo_Y", "residuo_T"]].copy()
salida_panel["tau"] = tau
salida_panel["tau_ci_lower"] = lb
salida_panel["tau_ci_upper"] = ub
salida_panel["significativo"] = (lb > 0) | (ub < 0)
salida_panel.to_csv(OUTPUT_DIR / "causal_forest_tau_panel.csv", index=False)
print(f"Guardado: {OUTPUT_DIR / 'causal_forest_tau_panel.csv'}  ({len(salida_panel)} filas)")


In [ ]:
# nivel distrito (promedio de X entre los años disponibles, un solo tau por distrito)
dist_X = merged.groupby(["ubigeo", "distrito", "departamento"])[cols_X].mean().reset_index()
n_anios = merged.groupby("ubigeo")["anio"].count().rename("n_anios_promediados")
dist_X = dist_X.merge(n_anios, on="ubigeo")

Xd = dist_X[cols_X].values
tau_d = cf.effect(Xd)
lb_d, ub_d = cf.effect_interval(Xd, alpha=0.05)

dist_X["tau"] = tau_d
dist_X["tau_ci_lower"] = lb_d
dist_X["tau_ci_upper"] = ub_d
dist_X["significativo"] = (lb_d > 0) | (ub_d < 0)

dist_X.to_csv(OUTPUT_DIR / "causal_forest_tau_distrital.csv", index=False)
print(f"Guardado: {OUTPUT_DIR / 'causal_forest_tau_distrital.csv'}  ({len(dist_X)} distritos)")
dist_X.sort_values("tau", ascending=False).head(10)
